In [ ]:
import os
import glob
import mne
import numpy as np
import gc

eeg_channels = ['Fz', 'Cz', 'C3', 'C4', 'Pz']

label_dict = {
    'sal': 0,
    'per_0.3': 1,
    'DZP0.2': 2
}

base_dir = "path/to/dataset"
output_dir = "output/folder/path"
os.makedirs(output_dir, exist_ok=True)

freqs = np.arange(1, 101, 1)
freqs2 = np.arange(3, 101, 1)

bands = {
    "delta": np.arange(1, 5, 1),
    "theta": np.arange(4, 9, 1),
    "alpha": np.arange(8, 14, 1),
    "beta": np.arange(13, 30, 1),
    "gamma": np.arange(30, 101, 1)
}

bands_itpc = {
    "delta": np.arange(3, 5, 1),   # 1 ve 2 Hz gg
    "theta": np.arange(4, 9, 1),
    "alpha": np.arange(8, 14, 1),
    "beta": np.arange(13, 30, 1),
    "gamma": np.arange(30, 101, 1)
}

# Değişken n_cycle
n_cycles = np.maximum(3, freqs / 2)
n_cycles2 = np.maximum(2, freqs2 / 4)

file_counter = 0

for label_name, class_id in label_dict.items():
    folder_path = os.path.join(base_dir, label_name)
    vhdr_files = glob.glob(f"{folder_path}/**/ASSR*.vhdr", recursive=True)

    if len(vhdr_files) == 0:
        print(f"Uyarı: {folder_path} içinde .vhdr dosyası bulunamadı")
        continue

    for file in vhdr_files:
        filename = os.path.basename(file)
        subject_name = os.path.basename(os.path.dirname(file))

        if "post" not in filename.lower():
            continue

        print(f"\n--> [{label_name}] - {subject_name} - {filename}")

        raw = mne.io.read_raw_brainvision(file, preload=True, verbose=False)

        try:
            raw.pick(eeg_channels)
        except ValueError:
            print(f"{filename} dosyasında belirtilen kanallar bulunamadı")
            continue

        raw.notch_filter(freqs=60.0, verbose=False)
        raw.filter(l_freq=1.0, h_freq=100.0, verbose=False)
        raw.resample(sfreq=500.0, verbose=False)

        events, event_id = mne.events_from_annotations(raw, verbose=False)
        sfreq = raw.info['sfreq']

        epochs = mne.Epochs(raw, events=events, event_id=event_id,
                            tmin=-0.1, tmax=1.0, baseline=(-0.1, 0.0),
                            preload=True, verbose=False)

        num_events = len(epochs)
        if num_events < 25:
            print(f"{filename} {num_events} event içeriyor, 25'ten az, atlanıyor")
            continue

        groups = []
        long_events = []

        # Dynamic Grouping
        chunk_size = 25
        start_indices = list(range(0, num_events - chunk_size + 1, chunk_size))

        if num_events % chunk_size != 0:
            start_indices.append(num_events - chunk_size)

        for idx in start_indices:
            groups.append(epochs[idx: idx + chunk_size])
            start_sample = events[idx, 0] - int(0.1 * sfreq)
            long_events.append([start_sample, 0, 1])

        long_events = np.array(long_events)

        long_epochs = mne.Epochs(
            raw,
            events=long_events,
            tmin=0,
            tmax=27.5 - (1 / sfreq),
            baseline=None,
            preload=True,
            verbose=False
        )

        tfr = mne.time_frequency.tfr_morlet(
            long_epochs, freqs=freqs, n_cycles=n_cycles,
            use_fft=True, return_itc=False, average=False,
            verbose=False, output="power"
        )
        #tfr.apply_baseline(baseline=(-0.1, 0.0), mode='logratio', verbose=False)

        power_total = mne.time_frequency.tfr_morlet(
            epochs,
            freqs=freqs2,
            n_cycles=n_cycles2,
            use_fft=True,
            return_itc=False,
            average=True,
            verbose=False,
        )

        evoked = epochs.average()

        power_evoked = mne.time_frequency.tfr_morlet(
            evoked,
            freqs=freqs2,
            n_cycles=n_cycles2,
            use_fft=True,
            return_itc=False,
            verbose=False,
        )
        power_evoked.data = power_evoked.data.astype(np.float32)

        power_induced = power_total.copy()
        power_induced.data = power_induced.data.astype(np.float32)
        power_induced.data -= power_evoked.data



        data = np.log1p(tfr.data).astype(np.float32)

        print(f"Toplam Epoch (uzun ve kısa gruplar): {len(long_epochs)}")

        original_filename = filename.replace(".vhdr", "")

        for epoch_idx, (single_long_epoch_data, group) in enumerate(zip(data, groups)):
            bandPLVvals = {}

            for name, band in bands_itpc.items():
                band_cycles = np.maximum(2, band / 4)
                itpc = mne.time_frequency.tfr_array_morlet(
                    group.get_data(),
                    sfreq=sfreq,
                    freqs=band,
                    n_cycles=band_cycles,
                    output="itc"
                )
                bandPLVvals[name] = itpc.astype(np.float32)

            file_name = f"{subject_name}_{original_filename}_epoch_{epoch_idx}.npz"
            save_path = os.path.join(output_dir, file_name)

            np.savez_compressed(
                save_path,
                evoked=power_evoked.data,
                induced=power_induced.data,
                total=power_total.data,
                delta=bandPLVvals["delta"],
                theta=bandPLVvals["theta"],
                alpha=bandPLVvals["alpha"],
                beta=bandPLVvals["beta"],
                gamma=bandPLVvals["gamma"],
                spect=single_long_epoch_data,
                label=class_id
            )

            file_counter += 1

        del raw, epochs, long_epochs, tfr, data, groups
        gc.collect()

print(f"\nToplam {file_counter} parça '{output_dir}' klasörüne kaydedildi.")